# 01. Dummy Classifier

Baseline for Logistic Regression, Random Forest, and MLP. The fixed `most_frequent` strategy predicts the most common training class for every student. It needs no hyperparameter search or feature preprocessing.

Use the shared `data/splits/splits.json` and metrics from `common/metrics.py`. Fit on Train and evaluate on Validation; reserve Test for the final project evaluation.

Running all cells saves summary, per-class scores, confusion matrix, and predictions under `reports/tables/dummy/`, plus `artifacts/dummy_pipeline.joblib` and its JSON metadata. Use a kernel with the project requirements installed.

## 1. Load data and validate the shared splits

CSV columns are separated by semicolons. Normalize CRLF line endings before hashing, matching `00_prepare_data.ipynb`. Split indices refer to the original CSV row positions. Validate all three partitions for invalid indices, duplicates, overlap, and full coverage.

In [ ]:
import hashlib
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

here = Path.cwd().resolve()
root = next((folder for folder in (here, *here.parents)
             if (folder / "common/preprocessing.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open the notebook from the project checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from common.metrics import (
    confusion_frame, evaluate, per_class_frame, summary_frame,
)
from common.preprocessing import FEATURE_COLUMNS, LABELS, TARGET

data_path = root / "data/raw/data.csv"
split_path = root / "data/splits/splits.json"
data = pd.read_csv(data_path, sep=";", encoding="utf-8-sig")
data.columns = data.columns.str.strip()
csv_bytes = data_path.read_bytes().replace(b"\r\n", b"\n")
data_hash = hashlib.sha256(csv_bytes).hexdigest()
split = json.loads(split_path.read_text(encoding="utf-8"))
if split["sha256"] != data_hash or split["n_rows"] != len(data):
    raise ValueError("CSV does not match the data used to create the shared splits.")

required_columns = set(FEATURE_COLUMNS) | {TARGET}
missing_columns = required_columns - set(data.columns)
if missing_columns or data.columns.duplicated().any():
    raise ValueError(f"Invalid CSV columns. Missing: {sorted(missing_columns)}")
if data[TARGET].isna().any() or not data[TARGET].isin(LABELS).all():
    raise ValueError("Target contains missing or unknown labels.")

def checked_rows(name):
    rows = split[name]
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"{name} must be a nonempty list of row indices.")
    if any(type(row) is not int or not 0 <= row < len(data) for row in rows):
        raise ValueError(f"{name} contains invalid row indices.")
    if len(set(rows)) != len(rows):
        raise ValueError(f"{name} contains duplicate row indices.")
    return rows

train_rows = checked_rows("train")
validation_rows = checked_rows("validation")
test_rows = checked_rows("test")
all_rows = train_rows + validation_rows + test_rows
if len(all_rows) != len(set(all_rows)):
    raise ValueError("Train, Validation, and Test must be disjoint.")
if set(all_rows) != set(range(len(data))):
    raise ValueError("The shared partitions must cover every CSV row.")

X = data.loc[:, list(FEATURE_COLUMNS)]
y = data[TARGET]
if set(y.iloc[train_rows]) != set(LABELS):
    raise ValueError("Train must contain all three project labels.")
print(f"CSV: {len(data):,} rows; Train: {len(train_rows):,}; Validation: {len(validation_rows):,}")
train_counts = y.iloc[train_rows].value_counts().reindex(LABELS, fill_value=0)
display(pd.DataFrame({"count": train_counts, "proportion": train_counts / len(train_rows)}))

## 2. Fit the baseline on Train

A one-step pipeline keeps the artifact interface consistent with the other model notebooks. The classifier ignores feature values. Its fitted class priors determine the majority class, with ties resolved by the classifier's class order.

In [ ]:
model = Pipeline([("model", DummyClassifier(strategy="most_frequent"))])
model.fit(X.iloc[train_rows], y.iloc[train_rows])
classifier = model.named_steps["model"]
predicted_label = classifier.classes_[np.argmax(classifier.class_prior_)]
print(f"Predicted class: {predicted_label}")
print(f"Training share: {classifier.class_prior_.max():.2%}")

## 3. Evaluate on Validation

Macro-F1 gives equal weight to each of the three classes. A majority-class baseline has zero recall for the other classes, even when accuracy appears reasonable. Shared metric functions assign zero precision when a class is never predicted.

For `most_frequent`, `predict_proba` returns probability 1 for the majority class and 0 for the other classes. These are fixed baseline outputs, not calibrated student risk estimates.

In [ ]:
X_val = X.iloc[validation_rows]
y_val = y.iloc[validation_rows]
y_pred = model.predict(X_val)
probabilities = pd.DataFrame(
    model.predict_proba(X_val), columns=model.classes_, index=X_val.index,
).reindex(columns=LABELS)
result = evaluate(y_val, y_pred)
summary = summary_frame({"Dummy (most_frequent)": result})
per_class = per_class_frame(result)
confusion = confusion_frame(result)

display(summary)
display(per_class)
display(confusion)
print(f"Validation rows: {result['n_samples']:,}")

## 4. Save reports and the fitted pipeline

The prediction table preserves original CSV row indices and names probability columns explicitly in the shared label order. Metadata records the input schema and fitted training priors. Reload the artifact and confirm that it reproduces the Validation predictions.

In [ ]:
table_dir = root / "reports/tables/dummy"
artifact_dir = root / "artifacts"
table_dir.mkdir(parents=True, exist_ok=True)
artifact_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(table_dir / "validation_summary.csv")
per_class.to_csv(table_dir / "validation_per_class.csv")
confusion.to_csv(table_dir / "validation_confusion.csv")
predictions = pd.DataFrame({
    "row_index": validation_rows,
    "y_true": y_val.to_numpy(),
    "y_pred": y_pred,
})
for label in LABELS:
    predictions[f"probability_{label}"] = probabilities[label].to_numpy()
predictions.to_csv(table_dir / "predictions_val.csv", index=False)

pipeline_path = artifact_dir / "dummy_pipeline.joblib"
joblib.dump(model, pipeline_path, compress=3)
reloaded = joblib.load(pipeline_path)
if not np.array_equal(reloaded.predict(X_val), y_pred):
    raise ValueError("Reloaded pipeline predictions differ from the fitted pipeline.")

metadata = {
    "model": "DummyClassifier",
    "strategy": "most_frequent",
    "predicted_class": str(predicted_label),
    "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "notebook": "notebooks/models/dummy/01_dummy_classifier.ipynb",
    "sklearn_version": sklearn.__version__,
    "data_sha256": data_hash,
    "split_seed": split["seed"],
    "labels": list(LABELS),
    "feature_columns": list(FEATURE_COLUMNS),
    "n_features_in": len(FEATURE_COLUMNS),
    "training_class_priors": {str(label): float(prior) for label, prior in
                             zip(classifier.classes_, classifier.class_prior_)},
    "n_train": len(train_rows),
    "n_validation": len(validation_rows),
    "validation_metrics": {key: float(summary.iloc[0][key]) for key in summary.columns},
}
(artifact_dir / "dummy_pipeline.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Saved:")
for path in (table_dir, pipeline_path, artifact_dir / "dummy_pipeline.json"):
    print(f"- {path.relative_to(root)}")
display(summary)

## Interpretation

Use the saved Validation summary as the minimum reference for the other models. Compare the same split and the same shared metrics, especially Macro-F1 and Dropout recall. No strategy is selected using Validation or Test results.